# Experiment Round 10: Training & Inference Pipeline
## exp2b_flash_learned_pool — Full 11M 3-LOB Pre-training + Embedding Generation

**Experiment:** `exp2b_flash_learned_pool` (Flash Attention + Learned Attention Pooling, no MoE)  
**Dataset:** ~11M members across Commercial, Medicare, Medicaid  
**Embedding dim:** 256  
**Architecture:** FlashAttentionTransformer with Learned Attention Pooling, SwiGLU, RoPE

### Table of Contents
1. **Environment Setup** — Imports, device config, path setup  
2. **Configuration** — Experiment parameters, optimization config  
3. **Data Loading** — BigQuery data ingestion, train/val split  
4. **Training** — `run_single_experiment` execution  
5. **Inference** — Load trained model, generate embeddings  
6. **Save Embeddings** — Export to BigQuery  
7. **Unit Tests** — Validate pipeline with synthetic data

### How to Use
1. Run cells 1-3 (setup, config, data) — ~15 min for BigQuery load
2. Run cell 4 (training) — GPU time depends on hardware (~hours on 4xT4)
3. Run cells 5-6 (inference + export) — ~30 min for embedding generation
4. Cell 7 (unit tests) — runs standalone, no BigQuery needed

## 1. Environment Setup

In [1]:
import sys
import os

# Ensure the notebook's working directory (dev/moe/) is on the import path.
# In Jupyter, __file__ is not defined, so we use os.getcwd().
# Run this notebook from the dev/moe/ directory where moe_flashattn_4_core.py lives.
MODULE_DIR = os.getcwd()
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

# Standard library
import gc
import time
import json
import copy
import threading
import warnings
from pathlib import Path
from typing import Dict, Optional, Tuple, List, Any

# Third-party
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.cloud import bigquery
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Core module imports — all configs, models, training, and utilities
# Source: dev/moe/moe_flashattn_4_core.py (unified single module)
# ---------------------------------------------------------------------------
from moe_flashattn_4_core import (
    # Configs
    BaseConfig,
    FlashAttentionConfig,
    MoEConfig,
    OptimizeConfig,
    # Experiment configs
    get_experiment_configs,
    # Data parsing utilities
    conv_cd,
    conv_age_gender,
    conv_lob,
    conv_target,
    ClinicalDataset,
    ClinicalDatasetLazy,
    create_collate_fn,
    # Model architecture
    FlashAttentionTransformer,
    FlashMoETransformer,
    BaselineTransformer,
    DataParallelWrapper,
    # Embedding extraction
    EmbeddingExtractor,
    # Training pipeline
    setup_experiment_logging,
    prepare_data_once,
    run_single_experiment,
    # Utilities
    cleanup_gpu_memory,
)

# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {props.total_memory / 1e9:.1f} GB")

Device: cuda
GPU count: 4
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB
  GPU 2: Tesla T4 — 15.6 GB
  GPU 3: Tesla T4 — 15.6 GB


## 2. Configuration

### Experiment Parameters
- **Experiment:** `exp2b_flash_learned_pool` — FlashAttention + Learned Attention Pooling (no MoE)
- **Embedding size:** 256
- **Epochs:** 1 (single pass over 11M; sufficient for convergence at this scale)
- **Experiment round:** `exp_round10_3lobs_formal_training`

### Optimizer Configuration
- **Scheduler:** Linear warmup (15%) → plateau (45%) → decay
- **Loss:** BCE with log-scaled pos_weight (max=200)
- **Optimizer:** AdamW (lr=2e-4, weight_decay=0.01 from BaseConfig defaults)

In [2]:
# ============================================================================
# EXPERIMENT CONFIGURATION
# ============================================================================

EXP_NAME = "exp2b_flash_learned_pool"
EXPERIMENT_ROUND = "exp_round10_3lobs_formal_training"
EMBEDDING_SIZE = 256
EPOCHS = 1

# Retrieve predefined architecture config for this experiment.
# exp2b uses: moe_config=None, use_learnt_att_pool=True
all_configs = get_experiment_configs()
moe_config, use_learnt_att_pool = all_configs[EXP_NAME]

print(f"Experiment:      {EXP_NAME}")
print(f"Round:           {EXPERIMENT_ROUND}")
print(f"Embedding size:  {EMBEDDING_SIZE}")
print(f"Epochs:          {EPOCHS}")
print(f"MoE config:      {moe_config}")
print(f"Learned pooling: {use_learnt_att_pool}")

Experiment:      exp2b_flash_learned_pool
Round:           exp_round10_3lobs_formal_training
Embedding size:  256
Epochs:          1
MoE config:      None
Learned pooling: True


In [25]:
# ============================================================================
# OPTIMIZATION CONFIGURATION
# ============================================================================

optimize_config = OptimizeConfig(
    scheduler_type="linear",       # Linear warmup + plateau + decay
    warmup_pct=0.15,               # First 15% of steps: linear warmup
    plateau_pct=0.45,              # Next 45% at peak LR (60% total before decay)
    min_lr_ratio=0.2,              # Decay to 20% of peak LR

    use_pos_weight=True,           # Frequency-based BCE weighting
    pos_weight_method="log_scaled",
    pos_weight_max=200,            # Cap weight for rare codes

    use_focal_loss=False,          # Standard BCE (no focal loss)
)

print(f"Scheduler:  {optimize_config.scheduler_type}")
print(f"Warmup:     {optimize_config.warmup_pct * 100:.0f}%")
print(f"Plateau:    {optimize_config.plateau_pct * 100:.0f}%")
print(f"Min LR:     {optimize_config.min_lr_ratio * 100:.0f}% of peak")
print(f"Loss:       BCE + pos_weight (method={optimize_config.pos_weight_method}, max={optimize_config.pos_weight_max})")

Scheduler:  linear
Warmup:     15%
Plateau:    45%
Min LR:     20% of peak
Loss:       BCE + pos_weight (method=log_scaled, max=200)


## 3. Data Loading & Preparation

### Data Source
- **BigQuery table:** `edp-prod-storage.edp_ent_sdoheir_cns.a834793_Combined_All_LOB_o3_train_ending`
- Contains ~11M members across Commercial, Medicare, Medicaid
- Each row = one member's longitudinal clinical record

### Pipeline
1. Load from BigQuery → `input_data` DataFrame
2. Deduplicate: keep only members with exactly 1 record → `df_unique`
3. Stratified train/val split (99%/1%) by LOB → `train_df`, `val_df`
4. Lazy dataset creation (memory-efficient) → `data_prepared_11M`

### Required Columns
| Column | Format | Example |
|--------|--------|---------|
| `individual_id` | string | `"MBR_001"` |
| `age_in_months` | `"val*val*..."` | `"360*361*362"` |
| `gender_cd` | `"val*val*..."` | `"1*1*1"` |
| `cd` | `"c1,c2*c3*..."` | `"100,200*300"` |
| `lob` | string | `"Commercial"` |
| `dt_cnt` | int | `45` |
| `target` | `"c1,c2*c3*..."` | `"50,60*70"` |

In [4]:
# ============================================================================
# STEP 1: LOAD TRAINING DATA FROM BIGQUERY
# ============================================================================

client = bigquery.Client()

training_sql = """
SELECT *
FROM `edp-prod-storage.edp_ent_sdoheir_cns.a834793_Combined_All_LOB_o3_train_ending`
limit 320
"""

print("Loading training data from BigQuery...")
print("Table: a834793_Combined_All_LOB_o3_train_ending")
start_time = time.time()

input_data = client.query(training_sql).to_dataframe()

elapsed = time.time() - start_time
print(f"Loaded {len(input_data):,} rows in {elapsed:.1f}s")
print(f"Columns: {list(input_data.columns)}")
print(f"Memory usage: {input_data.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Loading training data from BigQuery...
Table: a834793_Combined_All_LOB_o3_train_ending
Loaded 320 rows in 3.1s
Columns: ['individual_id', 'lob', 'index_dt', 'gender_cd', 'age_in_months', 'cd', 'target', 'dt_cnt']
Memory usage: 0.00 GB


In [5]:
# ============================================================================
# STEP 2: DEDUPLICATE — keep only members with exactly 1 record
# ============================================================================

member_counts = input_data.groupby("individual_id").size()
single_record_members = member_counts[member_counts == 1].index
df_unique = input_data[input_data["individual_id"].isin(single_record_members)].copy()

del input_data
gc.collect()

print(f"Unique members (single record): {len(df_unique):,}")
print(f"LOB distribution:\n{df_unique['lob'].value_counts()}")

# ============================================================================
# STEP 3: STRATIFIED TRAIN/VAL SPLIT (99% / 1%)
# ============================================================================

TRAIN_RATIO = 0.9
RANDOM_SEED = 42

train_df, val_df = train_test_split(
    df_unique,
    train_size=TRAIN_RATIO,
    stratify=df_unique["lob"],
    random_state=RANDOM_SEED,
)

gc.collect()

print(f"\nTrain: {len(train_df):,} | Val: {len(val_df):,}")
print(f"Train LOB:\n{train_df['lob'].value_counts()}")
print(f"Val LOB:\n{val_df['lob'].value_counts()}")

Unique members (single record): 320
LOB distribution:
lob
Medicaid    320
Name: count, dtype: int64

Train: 288 | Val: 32
Train LOB:
lob
Medicaid    288
Name: count, dtype: int64
Val LOB:
lob
Medicaid    32
Name: count, dtype: int64


In [6]:
# ============================================================================
# STEP 4: CREATE LAZY DATASETS + COMPUTE CODE FREQUENCIES
# ============================================================================
# ClinicalDatasetLazy stores raw strings and parses on-the-fly per __getitem__.
# Memory: ~130 GB for 11M (vs ~888 GB for eager ClinicalDataset).

data_prepared_11M = prepare_data_once(
    train_data=train_df,
    val_data=val_df,
    device=device,
    use_lazy=True,
)

gc.collect()
print(f"\n{data_prepared_11M}")


PREPARING DATA (ONE-TIME OPERATION)

[1/3] Creating training dataset (lazy)...
ClinicalDatasetLazy: Storing 288 samples as raw strings (lazy parsing)...
  Done in 0.0s. Estimated string memory: ~0.0 GB
  Parsing will happen on-the-fly in __getitem__ (parallelized by DataLoader workers)

[2/3] Creating validation dataset (lazy)...
ClinicalDatasetLazy: Storing 32 samples as raw strings (lazy parsing)...
  Done in 0.0s. Estimated string memory: ~0.0 GB
  Parsing will happen on-the-fly in __getitem__ (parallelized by DataLoader workers)

[3/3] Computing code frequencies...
  Computing code frequencies from 288 target strings...
  Found 527 unique codes

✅ Data preparation complete in 0.5s
   Train samples: 288
   Val samples: 32
   Unique codes: 527


PreparedData(train_samples=288, val_samples=32, code_frequencies_shape=(6297,))


## 4. Training

`run_single_experiment` handles the full training pipeline:
1. **Model creation** — FlashAttentionTransformer with learned pooling, auto-selects nhid/nhead
2. **Loss setup** — BCEWithLogitsLoss + log-scaled pos_weight from code frequencies
3. **Multi-GPU** — DataParallel wrapping when >1 GPU detected; LR scaled linearly
4. **Optimizer** — AdamW with OneCycle/Linear scheduler
5. **Training loop** — Per-epoch: train → evaluate → checkpoint → log metrics
6. **Final evaluation** — Comprehensive eval on last epoch (recall@K, NDCG, micro-recall, etc.)
7. **Model saving** — `.pt` checkpoint with `model_state_dict`, `config`, `moe_config`

**Outputs:**
- Training logs: `logs/{EXPERIMENT_ROUND}/{EXP_NAME}/training.log`
- Metrics: `logs/{EXPERIMENT_ROUND}/{EXP_NAME}/metrics.json`
- Saved model: `logs/{EXPERIMENT_ROUND}/{EXP_NAME}/saved_models/<model_name>_final.pt`
- Results JSON: `logs/{EXPERIMENT_ROUND}/{EXP_NAME}/final_results.json`

In [7]:
# ============================================================================
# RUN TRAINING
# ============================================================================

# cleanup_gpu_memory(verbose=True)
torch.cuda.empty_cache()

exp2b_baseline_results_11M = run_single_experiment(
    exp_name=EXP_NAME,
    moe_config=moe_config,
    use_learnt_att_pool=use_learnt_att_pool,
    prepared_data=data_prepared_11M,
    train_data=train_df,
    val_data=val_df,
    device=device,
    epochs=EPOCHS,
    experiment_round=EXPERIMENT_ROUND,
    embedding_size=EMBEDDING_SIZE,
    log_dir="logs",
    save_model=True,
    optimize_config=optimize_config,
)

16:37:29 - exp2b_flash_learned_pool - INFO - 
16:37:29 - exp2b_flash_learned_pool - INFO - EXPERIMENT: exp2b_flash_learned_pool
16:37:29 - exp2b_flash_learned_pool - INFO - ================================================================================
16:37:31 - exp2b_flash_learned_pool - INFO - Model: Flash Attention Transformer (FP16)
16:37:31 - exp2b_flash_learned_pool - INFO -   d_model=256, nhid=704, nhead=8
16:37:31 - exp2b_flash_learned_pool - INFO -   Daily Encoder: Learned Attention Pooling
16:37:31 - exp2b_flash_learned_pool - INFO - Total parameters: 25,325,209
16:37:31 - exp2b_flash_learned_pool - INFO - ✅ Using pre-prepared data
16:37:31 - exp2b_flash_learned_pool - INFO - Using BCEWithLogitsLoss
16:37:31 - exp2b_flash_learned_pool - INFO -   With pos_weight method: log_scaled
16:37:31 - exp2b_flash_learned_pool - INFO -  Enabling DataParallel with 4 GPUs
16:37:31 - exp2b_flash_learned_pool - INFO -    Per-GPU batch size: 32
16:37:31 - exp2b_flash_learned_pool - INFO -  

  Computing pos_weight using method: 'log_scaled'
  Log-scaled weights: min=1.00, max=200.00, mean=183.35, median=200.00
  Created: BCEWithLogitsLoss with pos_weight (log_scaled)
  Batch 0/2

🔍 GPU UTILIZATION CHECK (Batch 0):
   GPU 0: 0.09 GB allocated, 0.11 GB reserved
   GPU 1: 0.00 GB allocated, 0.00 GB reserved
   GPU 2: 0.00 GB allocated, 0.00 GB reserved
   GPU 3: 0.00 GB allocated, 0.00 GB reserved
    Loss: 0.8447 | R@10: 0.029 | R@20: 0.029 | μR@10: 0.003 | P@10: 0.003 | NDCG@20: 0.003 | PosBrier: 0.3206
    GPU 0: 0.69GB / 2.25GB peak
    GPU 1: 0.01GB / 1.92GB peak
    GPU 2: 0.01GB / 1.92GB peak
    GPU 3: 0.01GB / 1.92GB peak


16:37:38 - exp2b_flash_learned_pool - INFO -   Using batch-averaged training metrics (no re-evaluation)
16:37:38 - exp2b_flash_learned_pool - INFO -   Final epoch: Running comprehensive evaluation...



COMPREHENSIVE EVALUATION
Computing streaming metrics (memory-safe)...
  Processing batch 0/1...
Computing detailed metrics on 35 sampled predictions...
  Tier sizes: common=102, medium=154, rare=145, tail=126
Computing efficiency metrics...
Computing resource metrics...
💾 Saved: checkpoint_latest.pt


16:37:41 - exp2b_flash_learned_pool - INFO - 
--- Epoch 1 Summary ---
16:37:41 - exp2b_flash_learned_pool - INFO -   Train loss: 0.7845 → 0.7242
16:37:41 - exp2b_flash_learned_pool - INFO -   Val loss: 0.6817, Recall@10: 0.000, μRecall@10: 0.000, NDCG@20: 0.000
16:37:41 - exp2b_flash_learned_pool - INFO - 
Training completed in 10.3s
16:37:41 - exp2b_flash_learned_pool - INFO -   Using cached comprehensive evaluation from final epoch


✅ New best! Val loss: 0.0000
Model saved to: logs/exp_round10_3lobs_formal_training/exp2b_flash_learned_pool/saved_models/exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260323_163741_final.pt


16:37:41 - exp2b_flash_learned_pool - INFO - Model saved as: exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260323_163741
16:37:41 - exp2b_flash_learned_pool - INFO - 🗑️ Cleaned up 3 checkpoint files, freed 0.85 GB
16:37:41 - exp2b_flash_learned_pool - INFO - Complete results saved to None
16:37:41 - exp2b_flash_learned_pool - INFO - 
16:37:41 - exp2b_flash_learned_pool - INFO - EXPERIMENT COMPLETE: exp2b_flash_learned_pool
16:37:41 - exp2b_flash_learned_pool - INFO - ================================================================================
16:37:41 - exp2b_flash_learned_pool - INFO - Final Recall@10: 0.000
16:37:41 - exp2b_flash_learned_pool - INFO - Best Val Loss: 0.0000
16:37:41 - exp2b_flash_learned_pool - INFO - Training Time: 10.3s
16:37:41 - exp2b_flash_learned_pool - INFO - ================================================================================



Best model saved to: logs/exp_round10_3lobs_formal_training/exp2b_flash_learned_pool/saved_models/exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260323_163741_best.pt


In [8]:
# ============================================================================
# TRAINING RESULTS SUMMARY
# ============================================================================

print(f"{'=' * 70}")
print(f"TRAINING COMPLETE: {EXP_NAME}")
print(f"{'=' * 70}")
print(f"Model path: {exp2b_baseline_results_11M.get('model_path', 'N/A')}")
print(f"Model name: {exp2b_baseline_results_11M.get('model_name', 'N/A')}")
print(f"\nKey Metrics:")
for key in ["best_val_loss", "recall@10", "micro_recall@10", "ndcg@20", "training_time"]:
    val = exp2b_baseline_results_11M.get(key, "N/A")
    if isinstance(val, float):
        print(f"  {key}: {val:.4f}")
    else:
        print(f"  {key}: {val}")

# Store the model path for the inference section below.
TRAINED_MODEL_PATH = exp2b_baseline_results_11M.get("model_path")
print(f"\nTRAINED_MODEL_PATH = {TRAINED_MODEL_PATH}")
print("This variable is used in Section 5 (Inference) below.")

TRAINING COMPLETE: exp2b_flash_learned_pool
Model path: logs/exp_round10_3lobs_formal_training/exp2b_flash_learned_pool/saved_models/exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260323_163741_final.pt
Model name: exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260323_163741

Key Metrics:
  best_val_loss: N/A
  recall@10: 0.0000
  micro_recall@10: 0.0000
  ndcg@20: N/A
  training_time: N/A

TRAINED_MODEL_PATH = logs/exp_round10_3lobs_formal_training/exp2b_flash_learned_pool/saved_models/exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260323_163741_final.pt
This variable is used in Section 5 (Inference) below.


## 5. Inference — Embedding Generation

### How It Works
1. **Load checkpoint** — `load_model_from_checkpoint()` reconstructs the model from saved `config` and `model_state_dict`
2. **Build dataset** — `LazyClinicalDatasetInference` wraps the inference DataFrame (target column optional)
3. **Forward pass** — Run model on batches; `EmbeddingExtractor` hooks into `model.norm` to capture the temporal representation **before** the decoder head
4. **Patient embedding** — Extract the representation at the member's **last valid day** (`dt_cnt - 1`)

### Output
- `embeddings`: numpy array of shape `[num_members, 256]`
- `member_ids`: list of `individual_id` strings
- `index_dts`: list of `index_dt` strings

### Functions (derived from `moe_flashattn_3_lob3_downstream_running.ipynb`)
| Function | Purpose |
|----------|---------|
| `load_model_from_checkpoint` | Reconstruct model from `.pt` checkpoint |
| `LazyClinicalDatasetInference` | Memory-efficient dataset (no target required) |
| `generate_embeddings` | Main entry point — routes to single/multi-GPU |
| `_generate_embeddings_single_gpu` | Optimized single-GPU path |
| `_generate_embeddings_multi_gpu` | Parallel multi-GPU with ThreadPoolExecutor |

In [3]:
# ============================================================================
# MODEL LOADING FROM CHECKPOINT
# ============================================================================
# Source: dev/downstream/moe_flashattn_3_lob3_downstream_running.ipynb
#
# Reconstructs the full model architecture from checkpoint metadata and loads
# the saved weights.  Handles all three model families:
#   - BaselineTransformer
#   - FlashAttentionTransformer
#   - FlashMoETransformer (with automatic d_ff inference from expert weights)

def load_model_from_checkpoint(
    model_path: str,
    device: torch.device,
    verbose: bool = True,
) -> Tuple[torch.nn.Module, BaseConfig, Optional[MoEConfig], bool, str]:
    """
    Load a pretrained model from a .pt checkpoint.

    Checkpoint expected keys:
        model_state_dict, model_type, config, moe_config (optional)

    Returns:
        (model, config, moe_config, use_mixed_precision, model_type)
    """
    if verbose:
        print(f"\n{'=' * 70}")
        print(f"Loading model from: {model_path}")

    checkpoint_data = torch.load(model_path, map_location=device, weights_only=False)

    model_type = checkpoint_data.get("model_type", "Unknown")
    config_dict = checkpoint_data.get("config", {})
    moe_config_dict = checkpoint_data.get("moe_config", None)
    state_dict = checkpoint_data["model_state_dict"]

    if verbose:
        print(f"  Model type: {model_type}")
        print(f"  Embedding size: {config_dict.get('embedding_size', 256)}")
        print(f"  N layers: {config_dict.get('nlayers', 6)}")
        print(f"  Learned attention pooling: {config_dict.get('use_learnt_att_pool', False)}")

    # Infer learned pooling from state_dict keys (ground truth)
    use_learnt_att_pool_inferred = "daily_pooling.query" in state_dict

    # For MoE models, infer d_ff from expert weight shapes to avoid mismatch
    inferred_d_ff = None
    if "FlashMoE" in model_type:
        for key in state_dict.keys():
            if "experts.0.ffn.w_gate.weight" in key:
                d_ff_adjusted = state_dict[key].shape[0]
                inferred_d_ff = (d_ff_adjusted * 3 + 1) // 2
                if verbose:
                    print(f"  Inferred d_ff from expert weights: {inferred_d_ff}")
                break
        if inferred_d_ff is None:
            inferred_d_ff = config_dict.get("nhid", 512)

    # --- Reconstruct model by type ---
    moe_config_out = None

    if "FlashMoE" in model_type:
        config = FlashAttentionConfig(
            embedding_size=config_dict.get("embedding_size", 256),
            nhid=config_dict.get("nhid", 512),
            nhead=config_dict.get("nhead", 8),
            nlayers=config_dict.get("nlayers", 6),
            dropout=config_dict.get("dropout", 0.1),
            use_learnt_att_pool=use_learnt_att_pool_inferred,
            use_swiglu=config_dict.get("use_swiglu", True),
            use_rope=config_dict.get("use_rope", True),
            use_flash=config_dict.get("use_flash", True),
        )
        d_ff_to_use = inferred_d_ff or config_dict.get("nhid", 512)
        if moe_config_dict:
            if verbose and moe_config_dict.get("d_ff") != d_ff_to_use:
                print(f"  Correcting d_ff: checkpoint={moe_config_dict.get('d_ff')}, actual={d_ff_to_use}")
            moe_config_out = MoEConfig(
                d_model=moe_config_dict.get("d_model", config.embedding_size),
                d_ff=d_ff_to_use,
                num_experts=moe_config_dict.get("num_experts", 8),
                num_shared_experts=moe_config_dict.get("num_shared_experts", 1),
                top_k=moe_config_dict.get("top_k", 2),
                expert_dropout=moe_config_dict.get("expert_dropout", 0.1),
                load_balance_strategy=moe_config_dict.get("load_balance_strategy", "deepseek"),
                aux_loss_weight=moe_config_dict.get("aux_loss_weight", 0.001),
                use_moe_from_layer=moe_config_dict.get("use_moe_from_layer", 2),
                use_swiglu_experts=moe_config_dict.get("use_swiglu_experts", True),
                router_warmup_steps=moe_config_dict.get("router_warmup_steps", 0),
                z_loss_weight=moe_config_dict.get("z_loss_weight", 0.005),
                bias_lr=moe_config_dict.get("bias_lr", 1e-3),
                bias_momentum=moe_config_dict.get("bias_momentum", 0.6),
            )
        else:
            moe_config_out = MoEConfig(d_model=config.embedding_size, d_ff=config.nhid)
        model = FlashMoETransformer(config, moe_config_out)
        use_mixed_precision = True

    elif "FlashAttention" in model_type:
        config = FlashAttentionConfig(
            embedding_size=config_dict.get("embedding_size", 256),
            nhid=config_dict.get("nhid", 512),
            nhead=config_dict.get("nhead", 8),
            nlayers=config_dict.get("nlayers", 6),
            dropout=config_dict.get("dropout", 0.1),
            use_learnt_att_pool=use_learnt_att_pool_inferred,
            use_swiglu=config_dict.get("use_swiglu", True),
            use_rope=config_dict.get("use_rope", True),
            use_flash=config_dict.get("use_flash", True),
        )
        model = FlashAttentionTransformer(config)
        use_mixed_precision = True

    else:
        config = BaseConfig(
            embedding_size=config_dict.get("embedding_size", 256),
            nhid=config_dict.get("nhid", 512),
            nlayers=config_dict.get("nlayers", 6),
            dropout=config_dict.get("dropout", 0.1),
        )
        model = BaselineTransformer(config)
        use_mixed_precision = False

    model.load_state_dict(checkpoint_data["model_state_dict"])
    model = model.to(device)
    model.eval()

    if verbose:
        total_params = sum(p.numel() for p in model.parameters())
        print(f"  Model loaded successfully!")
        print(f"  Total parameters: {total_params:,}")
        print(f"  Mixed precision: {use_mixed_precision}")
        print(f"  Device: {device}")
        print(f"{'=' * 70}\n")

    return model, config, moe_config_out, use_mixed_precision, model_type

In [4]:
# ============================================================================
# LAZY CLINICAL DATASET FOR INFERENCE
# ============================================================================
# Source: dev/downstream/moe_flashattn_3_lob3_downstream_running.ipynb
#
# Same interface as ClinicalDatasetLazy but the target column is optional.
# When absent, dummy targets are returned so the collate_fn still works.

class LazyClinicalDatasetInference(Dataset):
    """Memory-efficient dataset for inference (target column optional)."""

    def __init__(self, df: pd.DataFrame, config: BaseConfig):
        self.config = config
        self.df = df.reset_index(drop=True)

        self.age_strs = self.df["age_in_months"].tolist()
        self.gender_strs = self.df["gender_cd"].tolist()
        self.cd_strs = self.df["cd"].tolist()
        self.lob_strs = self.df["lob"].tolist()
        self.dt_cnt = self.df["dt_cnt"].tolist()
        self.target_strs = (
            self.df["target"].tolist() if "target" in self.df.columns else None
        )

        print(f"LazyClinicalDatasetInference: {len(self.df):,} samples (lazy loading)")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        config = self.config

        age = torch.tensor(
            conv_age_gender(self.age_strs[idx], config.len_dy), dtype=torch.long
        )
        gender = torch.tensor(
            conv_age_gender(self.gender_strs[idx], config.len_dy, max_val=3),
            dtype=torch.long,
        )
        lob = torch.tensor(
            conv_lob(self.lob_strs[idx], config.len_dy), dtype=torch.long
        )
        codes = torch.tensor(
            conv_cd(self.cd_strs[idx], config.len_dy, config.len_cd), dtype=torch.long
        )

        if self.target_strs is not None:
            target_list = conv_target(
                self.target_strs[idx], config.len_dy, config.target_cd_cnt
            )
        else:
            target_list = [[0] for _ in range(config.len_dy)]

        return {
            "age": age,
            "gender": gender,
            "lob": lob,
            "codes": codes,
            "dt_cnt": self.dt_cnt[idx],
            "target": target_list,
        }

In [5]:
# ============================================================================
# EMBEDDING GENERATION FUNCTIONS
# ============================================================================
# Source: dev/downstream/moe_flashattn_3_lob3_downstream_running.ipynb
#
# generate_embeddings() is the public entry point.  It routes to single-GPU
# or multi-GPU depending on hardware and the multi_gpu flag.
#
# Optimisations:
#   - Pre-allocated pinned-memory output tensor (avoids np.vstack)
#   - Non-blocking GPU→CPU transfers
#   - torch.inference_mode (faster than no_grad)
#   - tqdm progress bar with throughput and ETA


def generate_embeddings(
    model: torch.nn.Module,
    config: BaseConfig,
    data: pd.DataFrame,
    device: torch.device,
    id_column: str = "individual_id",
    lob_value: Optional[str] = None,
    desc_prefix: str = "",
    batch_size: int = 64,
    num_workers: int = 4,
    use_mixed_precision: bool = True,
    verbose: bool = True,
    multi_gpu: bool = False,
    moe_config: Optional[MoEConfig] = None,
) -> Tuple[np.ndarray, List[str], List[str]]:
    """
    Generate member-level embeddings from a trained model.

    Args:
        model: Trained model in eval mode.
        config: Model configuration (BaseConfig / FlashAttentionConfig).
        data: DataFrame with clinical columns + individual_id + index_dt.
        device: Primary CUDA device.
        id_column: Column for member IDs (default 'individual_id').
        lob_value: If set, adds a 'lob' column when missing (e.g. 'Medicaid').
        desc_prefix: Progress bar label (e.g. 'Commercial').
        batch_size: Per-GPU batch size.
        num_workers: DataLoader workers.
        use_mixed_precision: Enable FP16 autocast for Flash models.
        verbose: Print progress.
        multi_gpu: Distribute across all visible GPUs.
        moe_config: Required for multi-GPU with MoE models.

    Returns:
        embeddings  — np.ndarray [num_members, embedding_size]
        member_ids  — list of member ID strings
        index_dts   — list of index date strings
    """
    start_time = time.time()
    n_samples = len(data)
    embedding_dim = config.embedding_size

    has_moe = (
        hasattr(model, "forward")
        and "return_moe_losses" in model.forward.__code__.co_varnames
    )
    n_gpus = torch.cuda.device_count() if multi_gpu else 1

    desc = f"{desc_prefix} " if desc_prefix else ""
    desc += "Embedding Generation"

    if verbose:
        print(f"\n{'=' * 70}")
        print(f"{desc.upper()}")
        print(f"{'=' * 70}")
        print(f"Samples: {n_samples:,} | Batch: {batch_size} | GPUs: {n_gpus}")
        print(f"Workers: {num_workers} | Mixed precision: {use_mixed_precision}")
        print(f"ID column: {id_column}")

    if lob_value and "lob" not in data.columns:
        data = data.copy()
        data["lob"] = lob_value
        if verbose:
            print(f"  Added 'lob'='{lob_value}' column")

    embeddings_output = torch.empty(
        (n_samples, embedding_dim), dtype=torch.float32, pin_memory=True
    )

    if id_column in data.columns:
        member_ids = data[id_column].astype(str).tolist()
    else:
        member_ids = data["individual_id"].astype(str).tolist()
        if verbose:
            print(f"  Warning: '{id_column}' not found, falling back to 'individual_id'")

    index_dts = data["index_dt"].astype(str).tolist()
    pbar_desc = (
        f"Generating {desc_prefix} embeddings" if desc_prefix else "Generating embeddings"
    )

    if n_gpus > 1 and multi_gpu:
        return _generate_embeddings_multi_gpu(
            model=model, config=config, data=data,
            embeddings_output=embeddings_output,
            member_ids=member_ids, index_dts=index_dts,
            n_gpus=n_gpus, batch_size=batch_size, num_workers=num_workers,
            use_mixed_precision=use_mixed_precision, has_moe=has_moe,
            moe_config=moe_config, verbose=verbose,
            start_time=start_time, pbar_desc=pbar_desc,
        )
    else:
        return _generate_embeddings_single_gpu(
            model=model, config=config, data=data, device=device,
            embeddings_output=embeddings_output,
            member_ids=member_ids, index_dts=index_dts,
            batch_size=batch_size, num_workers=num_workers,
            use_mixed_precision=use_mixed_precision, has_moe=has_moe,
            verbose=verbose, start_time=start_time, pbar_desc=pbar_desc,
        )


# ---------------------------------------------------------------------------
# Single-GPU path
# ---------------------------------------------------------------------------

def _generate_embeddings_single_gpu(
    model, config, data, device, embeddings_output,
    member_ids, index_dts, batch_size, num_workers,
    use_mixed_precision, has_moe, verbose, start_time,
    pbar_desc="Generating embeddings",
) -> Tuple[np.ndarray, List[str], List[str]]:
    """Optimised single-GPU embedding extraction."""
    n_samples = len(data)
    model.eval()

    dataset = LazyClinicalDatasetInference(data, config)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=create_collate_fn(config),
        num_workers=num_workers,
        pin_memory=True,
        prefetch_factor=2 if num_workers > 0 else None,
        persistent_workers=num_workers > 0,
    )

    current_idx = 0
    pbar = tqdm(dataloader, desc=pbar_desc, disable=not verbose)

    with torch.inference_mode():
        with EmbeddingExtractor(model) as extractor:
            for batch in pbar:
                batch_size_actual = batch["age"].shape[0]
                batch_start = current_idx
                batch_end = batch_start + batch_size_actual

                x = torch.cat(
                    [
                        batch["age"].unsqueeze(-1),
                        batch["gender"].unsqueeze(-1),
                        batch["lob"].unsqueeze(-1),
                        batch["codes"],
                    ],
                    dim=-1,
                ).to(device, non_blocking=True)

                dt_cnt = batch["dt_cnt"]

                if use_mixed_precision:
                    with torch.cuda.amp.autocast(dtype=torch.float16):
                        if has_moe:
                            _ = model(x, return_moe_losses=False)
                        else:
                            _ = model(x)
                else:
                    if has_moe:
                        _ = model(x, return_moe_losses=False)
                    else:
                        _ = model(x)

                dt_cnt_list = (
                    dt_cnt.tolist() if isinstance(dt_cnt, torch.Tensor) else dt_cnt
                )
                patient_embs = extractor.get_patient_embedding(dt_cnt_list)

                embeddings_output[batch_start:batch_end].copy_(
                    patient_embs.float(), non_blocking=True
                )
                current_idx = batch_end

                elapsed = time.time() - start_time
                speed = batch_end / elapsed
                eta = (n_samples - batch_end) / speed if speed > 0 else 0
                pbar.set_postfix({"speed": f"{speed:.0f}/s", "ETA": f"{eta:.0f}s"})

    if device.type == "cuda":
        torch.cuda.synchronize()

    embeddings = embeddings_output.numpy()
    elapsed = time.time() - start_time
    if verbose:
        print(f"\nComplete! Time: {elapsed:.1f}s | Speed: {n_samples / elapsed:,.0f} samples/s")
        print(f"   Output: {embeddings.shape}")

    return embeddings, member_ids, index_dts


# ---------------------------------------------------------------------------
# Multi-GPU path
# ---------------------------------------------------------------------------

def _resolve_per_gpu_num_workers(num_workers: int, n_gpus: int) -> int:
    """Resolve a notebook-safe worker count for each GPU chunk."""
    if num_workers <= 0:
        return 0
    return max(1, num_workers // n_gpus)

def _generate_embeddings_multi_gpu(
    model, config, data, embeddings_output, member_ids, index_dts,
    n_gpus, batch_size, num_workers, use_mixed_precision, has_moe,
    moe_config, verbose, start_time,
    pbar_desc="Multi-GPU",
) -> Tuple[np.ndarray, List[str], List[str]]:
    """Parallel multi-GPU embedding extraction via ThreadPoolExecutor."""
    n_samples = len(data)
    per_gpu_num_workers = _resolve_per_gpu_num_workers(num_workers, n_gpus)
    if verbose:
        print(f"Multi-GPU mode: {n_gpus} GPUs")
        print(f"Per-GPU num_workers: {per_gpu_num_workers}")

    # Deep-copy model to each GPU
    models = []
    for gpu_id in range(n_gpus):
        if verbose:
            print(f"  Cloning model to GPU {gpu_id}...")
        with torch.cuda.device(gpu_id):
            model_copy = copy.deepcopy(model)
            model_copy = model_copy.to(f"cuda:{gpu_id}")
            model_copy.eval()
            models.append(model_copy)

    # Split data evenly across GPUs
    chunk_size = (n_samples + n_gpus - 1) // n_gpus
    data_chunks = []
    start_indices = []
    for i in range(n_gpus):
        s = i * chunk_size
        e = min((i + 1) * chunk_size, n_samples)
        data_chunks.append(data.iloc[s:e].reset_index(drop=True))
        start_indices.append(s)
        if verbose:
            print(f"  GPU {i}: samples {s:,} to {e:,} ({e - s:,})")

    progress_lock = threading.Lock()
    total_processed = [0]
    errors = []

    def process_chunk(gpu_id, data_chunk, start_idx):
        if len(data_chunk) == 0:
            return
        try:
            gpu_device = torch.device(f"cuda:{gpu_id}")
            gpu_model = models[gpu_id]

            dataset = LazyClinicalDatasetInference(data_chunk, config)
            dataloader = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=create_collate_fn(config),
                num_workers=per_gpu_num_workers,
                pin_memory=True,
                prefetch_factor=2 if per_gpu_num_workers > 0 else None,
                persistent_workers=per_gpu_num_workers > 0,
            )

            local_idx = start_idx
            with torch.inference_mode():
                with EmbeddingExtractor(gpu_model) as extractor:
                    for batch in dataloader:
                        bs_actual = batch["age"].shape[0]
                        x = torch.cat(
                            [
                                batch["age"].unsqueeze(-1),
                                batch["gender"].unsqueeze(-1),
                                batch["lob"].unsqueeze(-1),
                                batch["codes"],
                            ],
                            dim=-1,
                        ).to(gpu_device, non_blocking=True)

                        dt_cnt = batch["dt_cnt"]
                        if use_mixed_precision:
                            with torch.cuda.amp.autocast(dtype=torch.float16):
                                if has_moe:
                                    _ = gpu_model(x, return_moe_losses=False)
                                else:
                                    _ = gpu_model(x)
                        else:
                            if has_moe:
                                _ = gpu_model(x, return_moe_losses=False)
                            else:
                                _ = gpu_model(x)

                        dt_cnt_list = (
                            dt_cnt.tolist()
                            if isinstance(dt_cnt, torch.Tensor)
                            else dt_cnt
                        )
                        patient_embs = extractor.get_patient_embedding(dt_cnt_list)

                        embeddings_output[local_idx : local_idx + bs_actual].copy_(
                            patient_embs.float(), non_blocking=True
                        )
                        local_idx += bs_actual

                        with progress_lock:
                            total_processed[0] += bs_actual

            torch.cuda.synchronize(gpu_device)
        except Exception as e:
            errors.append((gpu_id, str(e)))

    # Launch parallel workers
    if verbose:
        pbar = tqdm(total=n_samples, desc=f"{pbar_desc} ({n_gpus} GPUs)")

    with ThreadPoolExecutor(max_workers=n_gpus) as executor:
        futures = [
            executor.submit(process_chunk, gid, data_chunks[gid], start_indices[gid])
            for gid in range(n_gpus)
        ]
        last_count = 0
        while not all(f.done() for f in futures):
            with progress_lock:
                current = total_processed[0]
            if verbose:
                pbar.update(current - last_count)
            last_count = current
            time.sleep(0.1)

        if verbose:
            pbar.update(n_samples - last_count)
            pbar.close()

        for f in futures:
            f.result()

    if errors:
        raise RuntimeError(f"GPU errors: {errors}")

    for m in models:
        del m
    torch.cuda.empty_cache()

    embeddings = embeddings_output.numpy()
    elapsed = time.time() - start_time
    if verbose:
        print(f"\nComplete! Time: {elapsed:.1f}s | Speed: {n_samples / elapsed:,.0f} samples/s")
        print(f"   Effective: {n_samples / elapsed * n_gpus:,.0f} samples/s across {n_gpus} GPUs")
        print(f"   Output: {embeddings.shape}")

    return embeddings, member_ids, index_dts

### 5.1 Configure Model Path

Set the path to the trained model.  If you just ran training (Section 4), `TRAINED_MODEL_PATH` is already populated from the results dict.  Otherwise, set it manually below.

In [6]:
# ============================================================================
# MODEL PATH CONFIGURATION
# ============================================================================
# Option A — Automatically set from training above:
#   TRAINED_MODEL_PATH is already defined in the results summary cell.
#
# Option B — Manual override (uncomment and fill in):
TRAINED_MODEL_PATH = (
    "logs/exp_round10_3lobs_formal_training/"
    "exp2b_flash_learned_pool_formal/saved_models/" 
    "exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260312_095916_final.pt" # This is for new TE model with flash attention
)

print(f"Model path: {TRAINED_MODEL_PATH}")
assert os.path.exists(TRAINED_MODEL_PATH), f"Model file not found: {TRAINED_MODEL_PATH}"

Model path: logs/exp_round10_3lobs_formal_training/exp2b_flash_learned_pool_formal/saved_models/exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260312_095916_final.pt


In [7]:
# ============================================================================
# LOAD MODEL FOR INFERENCE
# ============================================================================

cleanup_gpu_memory(verbose=False)

model, config, moe_config_loaded, use_mixed_precision, model_type = load_model_from_checkpoint(
    model_path=TRAINED_MODEL_PATH,
    device=device,
    verbose=True,
)


Loading model from: logs/exp_round10_3lobs_formal_training/exp2b_flash_learned_pool_formal/saved_models/exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260312_095916_final.pt
  Model type: FlashAttentionTransformer
  Embedding size: 256
  N layers: 6
  Learned attention pooling: True
  Model loaded successfully!
  Total parameters: 25,325,209
  Mixed precision: True
  Device: cuda



### 5.2 Load Inference Data

Load the heldout commercial dataset for embedding generation.

**Data source:** 
- Experimental:`edp-prod-storage.edp_ent_sdoheir_cns.a964286_commercial_heldout_transformer_input_4_te_experiment_round_5`
    - **Sampling strategy:**
        - Before 2023-10-16 (in-time): 30% random sample
        - After 2023-10-16 (out-of-time): 100% — used for OOT validation
- Formal downstream eval:
    - Commercial: `edp-prod-storage.edp_ent_sdoheir_cns.a964286_commercial_embedding_raw_features_20241120_20250930`
    - Medicare: `edp-prod-storage.edp_ent_sdoheir_cns.a964286_medicare_embedding_raw_features_20240701_20250930` 
    

In [8]:
# ============================================================================
# LOAD COMMERCIAL HELDOUT DATA
# ============================================================================

client = bigquery.Client()

commercial_sql = """
SELECT *,
'Commercial' as lob
 FROM `edp-prod-storage.edp_ent_sdoheir_cns.a964286_commercial_embedding_raw_features_20241120_20250930`
"""

print("Loading commercial data from BigQuery with dt_cnt >= 10...")
df_cm = client.query(commercial_sql).to_dataframe()
print(f"Loaded {len(df_cm):,} rows")

Loading commercial data from BigQuery with dt_cnt >= 10...
Loaded 12,936,113 rows


##### Fold for formal downstream embedding generation

In [14]:
# ============================================================================
# This is for experimentation; not for formal downstream eval
# SAMPLE: 30% in-time, 100% out-of-time
# ============================================================================

OOT_CUTOFF = "2023-10-16"
df_cm["index_dt"] = pd.to_datetime(df_cm["index_dt"])

df_cm_before = df_cm[df_cm["index_dt"] <= pd.to_datetime(OOT_CUTOFF)]
df_cm_after = df_cm[df_cm["index_dt"] > pd.to_datetime(OOT_CUTOFF)]

df_cm_before_sample = df_cm_before.sample(frac=0.001, random_state=42)

df_cm_sample = pd.concat([df_cm_before_sample, df_cm_after])

print(f"\nSampling summary:")
print(f"  Before {OOT_CUTOFF}: {len(df_cm_before):,} -> {len(df_cm_before_sample):,} (30%)")
print(f"  After  {OOT_CUTOFF}: {len(df_cm_after):,} (100%, OOT)")
print(f"  Total inference set: {len(df_cm_sample):,}")

Loading commercial heldout data from BigQuery...
Loaded 6,840,066 rows

Sampling summary:
  Before 2023-10-16: 5,648,158 -> 1,694,447 (30%)
  After  2023-10-16: 1,191,908 (100%, OOT)
  Total inference set: 2,886,355


In [16]:
df_cm_before_sample = df_cm_before.sample(frac=0.001, random_state=42)

df_cm_sample = pd.concat([df_cm_before_sample, df_cm_after])

In [18]:
df_cm_before_sample.shape

(5648, 8)

In [33]:
df_cm.shape

(12936113, 6)

#### Generate embeddings

In [52]:
# ============================================================================
# GENERATE EMBEDDINGS
# ============================================================================

INFERENCE_BATCH_SIZE = 64

embeddings, individual_ids, index_dts = generate_embeddings(
    model=model,
    config=config,
    data=df_cm, # here should be df_cm_sample or df_cm
    device=device,
    id_column="individual_id",
    lob_value=None,            # lob column already in commercial data
    desc_prefix="Commercial",
    batch_size=INFERENCE_BATCH_SIZE,
    use_mixed_precision=use_mixed_precision,
    verbose=True,
    multi_gpu=True,
    moe_config=moe_config_loaded,
)

print(f"\nEmbedding matrix: {embeddings.shape}")
print(f"  Members:    {embeddings.shape[0]:,}")
print(f"  Dimensions: {embeddings.shape[1]}")
print(f"  dtype:      {embeddings.dtype}")


COMMERCIAL EMBEDDING GENERATION
Samples: 128 | Batch: 64 | GPUs: 4
Workers: 4 | Mixed precision: True
ID column: individual_id
Multi-GPU mode: 4 GPUs
  Cloning model to GPU 0...
  Cloning model to GPU 1...
  Cloning model to GPU 2...
  Cloning model to GPU 3...
  GPU 0: samples 0 to 32 (32)
  GPU 1: samples 32 to 64 (32)
  GPU 2: samples 64 to 96 (32)
  GPU 3: samples 96 to 128 (32)




Generating Commercial embeddings (4 GPUs):   0%|          | 0/128 [00:00<?, ?it/s]

LazyClinicalDatasetInference: 32 samples (lazy loading)
LazyClinicalDatasetInference: 32 samples (lazy loading)
LazyClinicalDatasetInference: 32 samples (lazy loading)


LazyClinicalDatasetInference: 32 samples (lazy loading)


Generating Commercial embeddings (4 GPUs):   0%|          | 0/128 [00:02<?, ?it/s]

Generating Commercial embeddings (4 GPUs):   0%|          | 0/128 [00:02<?, ?it/s]

Generating Commercial embeddings (4 GPUs):   0%|          | 0/128 [00:02<?, ?it/s]

Generating Commercial embeddings (4 GPUs):   0%|          | 0/128 [00:03<?, ?it/s]

Generating Commercial embeddings (4 GPUs):   0%|          | 0/128 [00:03<?, ?it/s]

Generating Commercial embeddings (4 GPUs): 100%|██████████| 128/128 [00:04<00:00, 29.02it/s]  


Complete! Time: 4.5s | Speed: 28 samples/s
   Effective: 113 samples/s across 4 GPUs
   Output: (128, 256)

Embedding matrix: (128, 256)
  Members:    128
  Dimensions: 256
  dtype:      float32


In [54]:
embeddings.shape

(128, 256)

## 6. Save Embeddings to BigQuery

Exports the embedding matrix to BigQuery.  Large uploads are automatically chunked (~500 MB per chunk) to stay within BigQuery memory limits.

**Output table schema:**
| Column | Type | Description |
|--------|------|-------------|
| `individual_id` | STRING | Member identifier |
| `index_dt` | STRING | Index date |
| `embedding_0` .. `embedding_255` | FLOAT | Embedding dimensions |
| `exp_name` | STRING | Experiment name metadata |
| `model_type` | STRING | Model type metadata |

In [45]:
# ============================================================================
# SAVE EMBEDDINGS TO BIGQUERY
# ============================================================================
# Source: dev/downstream/moe_flashattn_3_lob3_downstream_running.ipynb

def save_embeddings_to_bigquery(
    embeddings: np.ndarray,
    individual_ids: list,
    index_dts: list,
    project_id: str,
    dataset_id: str,
    table_name: str,
    exp_name: str = "",
    model_type: str = "",
    if_exists: str = "replace",
    max_bytes_per_chunk: int = 500_000_000,
) -> str:
    """
    Save embeddings to BigQuery with automatic chunking.

    Args:
        embeddings: [num_members, embedding_dim] numpy array
        individual_ids: member ID list
        index_dts: index date list
        project_id: GCP project (e.g. 'edp-prod-storage')
        dataset_id: BigQuery dataset (e.g. 'edp_ent_sdoheir_cns')
        table_name: target table name
        exp_name: experiment name for metadata column
        model_type: model type for metadata column
        if_exists: 'replace' | 'append' | 'fail'
        max_bytes_per_chunk: target chunk size in bytes

    Returns:
        Full BigQuery table path string.
    """
    from google.cloud import bigquery as bq

    n_total = len(individual_ids)
    embedding_dim = embeddings.shape[1]
    full_table_id = f"{project_id}.{dataset_id}.{table_name}"

    bytes_per_row = embedding_dim * 4 + 200
    chunk_size = max(1, max_bytes_per_chunk // bytes_per_row)
    n_chunks = (n_total + chunk_size - 1) // chunk_size

    print(f"Writing {n_total:,} rows to BigQuery: {full_table_id}")
    print(f"  Embedding dim: {embedding_dim} | Est. payload: {bytes_per_row * n_total / 1e9:.2f} GB")
    if n_chunks > 1:
        print(f"  Chunking: {n_chunks} uploads of ~{chunk_size:,} rows")

    bq_client = bq.Client()
    first_disposition = {
        "replace": bq.WriteDisposition.WRITE_TRUNCATE,
        "append": bq.WriteDisposition.WRITE_APPEND,
        "fail": bq.WriteDisposition.WRITE_EMPTY,
    }[if_exists]

    for chunk_idx in range(n_chunks):
        start = chunk_idx * chunk_size
        end = min(start + chunk_size, n_total)

        df_chunk = pd.DataFrame({
            "individual_id": individual_ids[start:end],
            "index_dt": index_dts[start:end],
        })
        for i in range(embedding_dim):
            df_chunk[f"embedding_{i}"] = embeddings[start:end, i].astype(np.float32)
        df_chunk["exp_name"] = exp_name
        df_chunk["model_type"] = model_type

        write_disp = first_disposition if chunk_idx == 0 else bq.WriteDisposition.WRITE_APPEND
        job_config = bq.LoadJobConfig(write_disposition=write_disp)
        job = bq_client.load_table_from_dataframe(df_chunk, full_table_id, job_config=job_config)
        job.result()

        print(f"  Chunk {chunk_idx + 1}/{n_chunks}: rows [{start:,}, {end:,}) uploaded")
        del df_chunk

    table = bq_client.get_table(full_table_id)
    print(f"Loaded {table.num_rows:,} rows to {full_table_id}")
    return full_table_id

In [46]:
print("embedding generation done")

embedding generation done


In [48]:
# ============================================================================
# EXPORT EMBEDDINGS
# ============================================================================

PROJECT_ID = "edp-prod-storage"
DATASET_ID = "edp_ent_sdoheir_cns"
TABLE_NAME = "a964286_exp_round10_exp2b_commercial_embeddings_20241120_20250930"

full_table_path = save_embeddings_to_bigquery(
    embeddings=embeddings,
    individual_ids=individual_ids,
    index_dts=index_dts,
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_name=TABLE_NAME,
    exp_name=EXP_NAME,
    model_type=model_type,
    if_exists="replace",
)

print(f"\nEmbeddings saved to: {full_table_path}")

Writing 64 rows to BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_exp_round10_exp2b_commercial_embeddings_20241120_20250930
  Embedding dim: 256 | Est. payload: 0.00 GB
  Chunk 1/1: rows [0, 64) uploaded
Loaded 64 rows to edp-prod-storage.edp_ent_sdoheir_cns.a964286_exp_round10_exp2b_commercial_embeddings_20241120_20250930

Embeddings saved to: edp-prod-storage.edp_ent_sdoheir_cns.a964286_exp_round10_exp2b_commercial_embeddings_20241120_20250930


In [ ]:
print("embedding saved")

### Split embedding generation for large member size

In [15]:
# ============================================================================
# FULL DATASET SHARDED EMBEDDING GENERATION (4 PARTS + LOCAL CHECKPOINTS)
# ============================================================================
# This cell intentionally uses df_cm (full heldout dataset), not df_cm_sample.
# It runs one shard at a time, saves each shard locally, and clears memory
# between shards. You can reload the saved .npz files later and then write the
# combined result to BigQuery in a separate step.

N_SHARDS = 4
SHARDED_MULTI_GPU = True  # Recommended for reliability. Set True only after validating one shard.
if SHARDED_MULTI_GPU:
    INFERENCE_BATCH_SIZE = 16 # set it to 16 when using multi-GPU 
else:
    INFERENCE_BATCH_SIZE = 64
SHARDED_BATCH_SIZE = INFERENCE_BATCH_SIZE
SHARDED_NUM_WORKERS = 0  # Notebook-safe setting for repeated shard inference loops.
LOCAL_SHARD_DIR = Path("embedding_output/exp_round10_3lobs_formal_eval_commercial_ip")
LOCAL_SHARD_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = LOCAL_SHARD_DIR / f"{EXP_NAME}_commercial_shard_manifest.json"

def save_local_embedding_shard(
    file_path: Path,
    embeddings: np.ndarray,
    individual_ids: List[str],
    index_dts: List[str],
    shard_name: str,
    shard_index: int,
    total_shards: int,
    exp_name: str,
    model_type: str,
    source_rows: int,
    batch_size: int,
    multi_gpu: bool,
    data_source_name: str,
    start_row: int,
    end_row: int,
    generated_at: str,
 ) -> None:
    """Save one embedding shard locally as a compressed NPZ checkpoint."""
    np.savez_compressed(
        file_path,
        embeddings=embeddings.astype(np.float32),
        individual_ids=np.asarray(individual_ids, dtype=str),
        index_dts=np.asarray(index_dts, dtype=str),
        shard_name=shard_name,
        shard_index=shard_index,
        total_shards=total_shards,
        exp_name=exp_name,
        model_type=model_type,
        source_rows=source_rows,
        embedding_dim=embeddings.shape[1],
        batch_size=batch_size,
        multi_gpu=multi_gpu,
        data_source_name=data_source_name,
        start_row=start_row,
        end_row=end_row,
        generated_at=generated_at,
    )

if MANIFEST_PATH.exists():
    shard_manifest = json.loads(MANIFEST_PATH.read_text())
else:
    shard_manifest = []

manifest_by_name = {entry["shard_name"]: entry for entry in shard_manifest}

n_total_rows = len(df_cm)
rows_per_shard = (n_total_rows + N_SHARDS - 1) // N_SHARDS
print(f"Full df_cm rows: {n_total_rows:,}")
print(f"Shard count: {N_SHARDS}")
print(f"Rows per shard (max): {rows_per_shard:,}")
print(f"Shard output directory: {LOCAL_SHARD_DIR.resolve()}")
print(f"Shard batch size: {SHARDED_BATCH_SIZE}")
print(f"Shard multi_gpu: {SHARDED_MULTI_GPU}")
print(f"Shard num_workers: {SHARDED_NUM_WORKERS}")

Full df_cm rows: 12,936,113
Shard count: 4
Rows per shard (max): 3,234,029
Shard output directory: /home/jupyter/ClinTE/Clinical_Transformer_Emb/model_refactor/embedding_output/exp_round10_3lobs_formal_eval_commercial_ip
Shard batch size: 16
Shard multi_gpu: True
Shard num_workers: 0


#### Test for loop

In [14]:
# ============================================================================
# SMOKE TEST FOR THE SHARDED LOOP (NO SAVING)
# ============================================================================
# Uses a small sample from df_cm so you can verify that embeddings are generated
# before running the full shard loop above.
#
# Note: num_workers is set to 0 here to avoid notebook multiprocessing cleanup
# errors after each tiny shard finishes.

TEST_SAMPLE_ROWS = min(1024, len(df_cm))
TEST_N_SHARDS = 4
TEST_BATCH_SIZE = min(16, SHARDED_BATCH_SIZE)
TEST_MULTI_GPU = True
TEST_NUM_WORKERS = 0

df_cm_test = df_cm.sample(n=TEST_SAMPLE_ROWS, random_state=42).reset_index(drop=True)
test_rows_per_shard = (len(df_cm_test) + TEST_N_SHARDS - 1) // TEST_N_SHARDS

test_embedding_parts = []
test_individual_ids = []
test_index_dts = []

print(f"Test sample rows: {len(df_cm_test):,}")
print(f"Test shards: {TEST_N_SHARDS}")
print(f"Test batch size: {TEST_BATCH_SIZE}")
print(f"Test multi_gpu: {TEST_MULTI_GPU}")
print(f"Test num_workers: {TEST_NUM_WORKERS}")


for shard_idx in range(TEST_N_SHARDS):
    start_row = shard_idx * test_rows_per_shard
    end_row = min(start_row + test_rows_per_shard, len(df_cm_test))
    if start_row >= end_row:
        continue

    shard_number = shard_idx + 1
    shard_df_test = df_cm_test.iloc[start_row:end_row].copy()

    print("\n" + "-" * 70)
    print(f"Test shard {shard_number}/{TEST_N_SHARDS}: rows [{start_row}, {end_row})")

    shard_embeddings, shard_ids, shard_dates = generate_embeddings(
        model=model,
        config=config,
        data=shard_df_test,
        device=device,
        id_column="individual_id",
        lob_value=None,
        desc_prefix=f"Commercial test shard {shard_number}",
        batch_size=TEST_BATCH_SIZE,
        num_workers=TEST_NUM_WORKERS,
        use_mixed_precision=use_mixed_precision,
        verbose=True,
        multi_gpu=TEST_MULTI_GPU,
        moe_config=moe_config_loaded,
    )

    print(f"Shard embedding shape: {shard_embeddings.shape}")

    test_embedding_parts.append(shard_embeddings)
    test_individual_ids.extend(shard_ids)
    test_index_dts.extend(shard_dates)

    del shard_df_test, shard_embeddings, shard_ids, shard_dates
    gc.collect()
    cleanup_gpu_memory(verbose=False)
    if device.type == "cuda":
        torch.cuda.empty_cache()

test_embeddings = np.vstack(test_embedding_parts)

print("\n" + "=" * 70)
print(f"Combined test embedding shape: {test_embeddings.shape}")
print(f"Test IDs: {len(test_individual_ids):,}")
print(f"Test index dates: {len(test_index_dts):,}")
print("First 3 test IDs:", test_individual_ids[:3])
print("First 3 test index dates:", test_index_dts[:3])

Test sample rows: 1,024
Test shards: 4
Test batch size: 16
Test multi_gpu: True
Test num_workers: 0

----------------------------------------------------------------------
Test shard 1/4: rows [0, 256)

COMMERCIAL TEST SHARD 1 EMBEDDING GENERATION
Samples: 256 | Batch: 16 | GPUs: 4
Workers: 0 | Mixed precision: True
ID column: individual_id
Multi-GPU mode: 4 GPUs
Per-GPU num_workers: 0
  Cloning model to GPU 0...
  Cloning model to GPU 1...
  Cloning model to GPU 2...
  Cloning model to GPU 3...
  GPU 0: samples 0 to 64 (64)
  GPU 1: samples 64 to 128 (64)
  GPU 2: samples 128 to 192 (64)
  GPU 3: samples 192 to 256 (64)


Generating Commercial test shard 1 embeddings (4 GPUs):   0%|          | 0/256 [00:00<?, ?it/s]

LazyClinicalDatasetInference: 64 samples (lazy loading)
LazyClinicalDatasetInference: 64 samples (lazy loading)
LazyClinicalDatasetInference: 64 samples (lazy loading)
LazyClinicalDatasetInference: 64 samples (lazy loading)


Generating Commercial test shard 1 embeddings (4 GPUs): 100%|██████████| 256/256 [00:01<00:00, 152.58it/s]



Complete! Time: 1.8s | Speed: 140 samples/s
   Effective: 558 samples/s across 4 GPUs
   Output: (256, 256)
Shard embedding shape: (256, 256)

----------------------------------------------------------------------
Test shard 2/4: rows [256, 512)

COMMERCIAL TEST SHARD 2 EMBEDDING GENERATION
Samples: 256 | Batch: 16 | GPUs: 4
Workers: 0 | Mixed precision: True
ID column: individual_id
Multi-GPU mode: 4 GPUs
Per-GPU num_workers: 0
  Cloning model to GPU 0...
  Cloning model to GPU 1...
  Cloning model to GPU 2...
  Cloning model to GPU 3...
  GPU 0: samples 0 to 64 (64)
  GPU 1: samples 64 to 128 (64)
  GPU 2: samples 128 to 192 (64)
  GPU 3: samples 192 to 256 (64)


...


Complete! Time: 1.8s | Speed: 141 samples/s
   Effective: 564 samples/s across 4 GPUs
   Output: (256, 256)
Shard embedding shape: (256, 256)

----------------------------------------------------------------------
Test shard 4/4: rows [768, 1024)

COMMERCIAL TEST SHARD 4 EMBEDDING GENERATION
Samples: 256 | Batch: 16 | GPUs: 4
Workers: 0 | Mixed precision: True
ID column: individual_id
Multi-GPU mode: 4 GPUs
Per-GPU num_workers: 0
  Cloning model to GPU 0...
  Cloning model to GPU 1...
  Cloning model to GPU 2...
  Cloning model to GPU 3...
  GPU 0: samples 0 to 64 (64)
  GPU 1: samples 64 to 128 (64)
  GPU 2: samples 128 to 192 (64)
  GPU 3: samples 192 to 256 (64)


Generating Commercial test shard 4 embeddings (4 GPUs):   0%|          | 0/256 [00:00<?, ?it/s]

LazyClinicalDatasetInference: 64 samples (lazy loading)
LazyClinicalDatasetInference: 64 samples (lazy loading)
LazyClinicalDatasetInference: 64 samples (lazy loading)
LazyClinicalDatasetInference: 64 samples (lazy loading)


Generating Commercial test shard 4 embeddings (4 GPUs): 100%|██████████| 256/256 [00:01<00:00, 156.21it/s]



Complete! Time: 1.8s | Speed: 143 samples/s
   Effective: 573 samples/s across 4 GPUs
   Output: (256, 256)
Shard embedding shape: (256, 256)

Combined test embedding shape: (1024, 256)
Test IDs: 1,024
Test index dates: 1,024
First 3 test IDs: ['3853250818258', '46094114401', '8835836736874']
First 3 test index dates: ['2025-02-16', '2025-06-16', '2024-12-16']


#### Formal run

In [17]:
from tqdm.notebook import tqdm
for shard_idx in tqdm(range(N_SHARDS)):
    start_row = shard_idx * rows_per_shard
    end_row = min(start_row + rows_per_shard, n_total_rows)
    if start_row >= end_row:
        continue

    shard_number = shard_idx + 1
    shard_name = f"part_{shard_number:02d}_of_{N_SHARDS:02d}"
    shard_file = LOCAL_SHARD_DIR / f"{EXP_NAME}_commercial_ip_11M_{shard_name}.npz"
    expected_rows = end_row - start_row

    print("\n" + "=" * 70)
    print(f"Processing shard {shard_name}: rows [{start_row:,}, {end_row:,})")
    print(f"Expected rows: {expected_rows:,}")
    print(f"Target file: {shard_file}")

    if shard_file.exists():
        print(f"Shard already exists. Skipping generation: {shard_file}")
        manifest_by_name[shard_name] = {
            "shard_name": shard_name,
            "shard_index": shard_number,
            "rows_expected": expected_rows,
            "rows_written": expected_rows,
            "file_path": str(shard_file.resolve()),
            "status": "skipped_existing",
            "start_row": start_row,
            "end_row": end_row,
        }
        MANIFEST_PATH.write_text(json.dumps(list(manifest_by_name.values()), indent=2))
        continue

    shard_df_cm = df_cm.iloc[start_row:end_row].copy()
    shard_started_at = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"Shard started at {shard_started_at}")
    shard_embeddings, shard_individual_ids, shard_index_dts = generate_embeddings(
        model=model,
        config=config,
        data=shard_df_cm,
        device=device,
        id_column="individual_id",
        lob_value=None,
        desc_prefix=f"Commercial {shard_name}",
        batch_size=SHARDED_BATCH_SIZE,
        use_mixed_precision=use_mixed_precision,
        verbose=True,
        multi_gpu=SHARDED_MULTI_GPU,
        moe_config=moe_config_loaded,
        num_workers=SHARDED_NUM_WORKERS,
    )
    shard_ended_at = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"Embedding generated complete at {shard_ended_at}")
    assert shard_embeddings.shape[0] == expected_rows, (
        f"Embedding row count mismatch for {shard_name}: "
        f"expected {expected_rows}, got {shard_embeddings.shape[0]}"
    )
    assert len(shard_individual_ids) == expected_rows, (
        f"ID row count mismatch for {shard_name}: "
        f"expected {expected_rows}, got {len(shard_individual_ids)}"
    )
    assert len(shard_index_dts) == expected_rows, (
        f"Date row count mismatch for {shard_name}: "
        f"expected {expected_rows}, got {len(shard_index_dts)}"
    )

    save_local_embedding_shard(
        file_path=shard_file,
        embeddings=shard_embeddings,
        individual_ids=shard_individual_ids,
        index_dts=shard_index_dts,
        shard_name=shard_name,
        shard_index=shard_number,
        total_shards=N_SHARDS,
        exp_name=EXP_NAME,
        model_type=model_type,
        source_rows=expected_rows,
        batch_size=SHARDED_BATCH_SIZE,
        multi_gpu=SHARDED_MULTI_GPU,
        data_source_name="df_cm",
        start_row=start_row,
        end_row=end_row,
        generated_at=shard_started_at,
    )

    manifest_by_name[shard_name] = {
        "shard_name": shard_name,
        "shard_index": shard_number,
        "rows_expected": expected_rows,
        "rows_written": int(shard_embeddings.shape[0]),
        "file_path": str(shard_file.resolve()),
        "status": "completed",
        "start_row": start_row,
        "end_row": end_row,
        "generated_at": shard_started_at,
    }
    MANIFEST_PATH.write_text(json.dumps(list(manifest_by_name.values()), indent=2))

    print(f"Saved shard {shard_name} to: {shard_file}")
    print(f"Saved rows: {shard_embeddings.shape[0]:,}")

    del shard_df_cm, shard_embeddings, shard_individual_ids, shard_index_dts
    gc.collect()
    cleanup_gpu_memory(verbose=False)
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("Shard generation complete.")
print(f"Manifest written to: {MANIFEST_PATH.resolve()}")
print(f"Shard files written to: {LOCAL_SHARD_DIR.resolve()}")
print("You can reload the saved .npz shards later and manually combine them before exporting to BigQuery.")

  0%|          | 0/4 [00:00<?, ?it/s]


Processing shard part_01_of_04: rows [0, 3,234,029)
Expected rows: 3,234,029
Target file: embedding_output/exp_round10_3lobs_formal_eval_commercial_ip/exp2b_flash_learned_pool_commercial_ip_11M_part_01_of_04.npz
Shard started at 2026-04-07 17:15:55

COMMERCIAL PART_01_OF_04 EMBEDDING GENERATION
Samples: 3,234,029 | Batch: 16 | GPUs: 4
Workers: 0 | Mixed precision: True
ID column: individual_id
Multi-GPU mode: 4 GPUs
Per-GPU num_workers: 0
  Cloning model to GPU 0...
  Cloning model to GPU 1...
  Cloning model to GPU 2...
  Cloning model to GPU 3...
  GPU 0: samples 0 to 808,508 (808,508)
  GPU 1: samples 808,508 to 1,617,016 (808,508)
  GPU 2: samples 1,617,016 to 2,425,524 (808,508)
  GPU 3: samples 2,425,524 to 3,234,029 (808,505)


Generating Commercial part_01_of_04 embeddings (4 GPUs):   0%|          | 0/3234029 [00:00<?, ?it/s]

LazyClinicalDatasetInference: 808,508 samples (lazy loading)
LazyClinicalDatasetInference: 808,505 samples (lazy loading)
LazyClinicalDatasetInference: 808,508 samples (lazy loading)
LazyClinicalDatasetInference: 808,508 samples (lazy loading)

Complete! Time: 25533.4s | Speed: 127 samples/s
   Effective: 507 samples/s across 4 GPUs
   Output: (3234029, 256)
Embedding generated complete at 2026-04-08 00:21:29
Saved shard part_01_of_04 to: embedding_output/exp_round10_3lobs_formal_eval_commercial_ip/exp2b_flash_learned_pool_commercial_ip_11M_part_01_of_04.npz
Saved rows: 3,234,029

Processing shard part_02_of_04: rows [3,234,029, 6,468,058)
Expected rows: 3,234,029
Target file: embedding_output/exp_round10_3lobs_formal_eval_commercial_ip/exp2b_flash_learned_pool_commercial_ip_11M_part_02_of_04.npz
Shard started at 2026-04-08 00:24:33

COMMERCIAL PART_02_OF_04 EMBEDDING GENERATION
Samples: 3,234,029 | Batch: 16 | GPUs: 4
Workers: 0 | Mixed precision: True
ID column: individual_id
Multi-G

Generating Commercial part_02_of_04 embeddings (4 GPUs):   0%|          | 0/3234029 [00:00<?, ?it/s]

LazyClinicalDatasetInference: 808,508 samples (lazy loading)
LazyClinicalDatasetInference: 808,505 samples (lazy loading)
LazyClinicalDatasetInference: 808,508 samples (lazy loading)
LazyClinicalDatasetInference: 808,508 samples (lazy loading)

Complete! Time: 25755.4s | Speed: 126 samples/s
   Effective: 502 samples/s across 4 GPUs
   Output: (3234029, 256)
Embedding generated complete at 2026-04-08 07:33:49
Saved shard part_02_of_04 to: embedding_output/exp_round10_3lobs_formal_eval_commercial_ip/exp2b_flash_learned_pool_commercial_ip_11M_part_02_of_04.npz
Saved rows: 3,234,029

Processing shard part_03_of_04: rows [6,468,058, 9,702,087)
Expected rows: 3,234,029
Target file: embedding_output/exp_round10_3lobs_formal_eval_commercial_ip/exp2b_flash_learned_pool_commercial_ip_11M_part_03_of_04.npz
Shard started at 2026-04-08 07:36:55

COMMERCIAL PART_03_OF_04 EMBEDDING GENERATION
Samples: 3,234,029 | Batch: 16 | GPUs: 4
Workers: 0 | Mixed precision: True
ID column: individual_id
Multi-G

Generating Commercial part_03_of_04 embeddings (4 GPUs):   0%|          | 0/3234029 [00:00<?, ?it/s]

LazyClinicalDatasetInference: 808,508 samples (lazy loading)
LazyClinicalDatasetInference: 808,505 samples (lazy loading)
LazyClinicalDatasetInference: 808,508 samples (lazy loading)
LazyClinicalDatasetInference: 808,508 samples (lazy loading)

Complete! Time: 25643.5s | Speed: 126 samples/s
   Effective: 504 samples/s across 4 GPUs
   Output: (3234029, 256)
Embedding generated complete at 2026-04-08 14:44:19
Saved shard part_03_of_04 to: embedding_output/exp_round10_3lobs_formal_eval_commercial_ip/exp2b_flash_learned_pool_commercial_ip_11M_part_03_of_04.npz
Saved rows: 3,234,029

Processing shard part_04_of_04: rows [9,702,087, 12,936,113)
Expected rows: 3,234,026
Target file: embedding_output/exp_round10_3lobs_formal_eval_commercial_ip/exp2b_flash_learned_pool_commercial_ip_11M_part_04_of_04.npz
Shard started at 2026-04-08 14:47:25

COMMERCIAL PART_04_OF_04 EMBEDDING GENERATION
Samples: 3,234,026 | Batch: 16 | GPUs: 4
Workers: 0 | Mixed precision: True
ID column: individual_id
Multi-

Generating Commercial part_04_of_04 embeddings (4 GPUs):   0%|          | 0/3234026 [00:00<?, ?it/s]

LazyClinicalDatasetInference: 808,507 samples (lazy loading)
LazyClinicalDatasetInference: 808,507 samples (lazy loading)
LazyClinicalDatasetInference: 808,505 samples (lazy loading)
LazyClinicalDatasetInference: 808,507 samples (lazy loading)

Complete! Time: 26850.1s | Speed: 120 samples/s
   Effective: 482 samples/s across 4 GPUs
   Output: (3234026, 256)
Embedding generated complete at 2026-04-08 22:14:56
Saved shard part_04_of_04 to: embedding_output/exp_round10_3lobs_formal_eval_commercial_ip/exp2b_flash_learned_pool_commercial_ip_11M_part_04_of_04.npz
Saved rows: 3,234,026

Shard generation complete.
Manifest written to: /home/jupyter/ClinTE/Clinical_Transformer_Emb/model_refactor/embedding_output/exp_round10_3lobs_formal_eval_commercial_ip/exp2b_flash_learned_pool_commercial_shard_manifest.json
Shard files written to: /home/jupyter/ClinTE/Clinical_Transformer_Emb/model_refactor/embedding_output/exp_round10_3lobs_formal_eval_commercial_ip
You can reload the saved .npz shards lat

In [ ]:
# ============================================================================
# LOAD LOCAL SHARDS + WRITE TO BIGQUERY
# ============================================================================
# Reads the .npz shard format written by save_local_embedding_shard(...) and
# writes all four shards into one BigQuery table in shard order.

assert "save_local_embedding_shard" in globals(), (
    "Run the shard-generation helper cell first so the shard NPZ schema is defined."
 )
assert "save_embeddings_to_bigquery" in globals(), (
    "Run the BigQuery export helper cell first so the upload function is available."
 )

EXPORT_PROJECT_ID = "edp-prod-storage"
EXPORT_DATASET_ID = "edp_ent_sdoheir_cns"
EXPORT_TABLE_NAME = "a964286_exp_round10_exp2b_commercial_embeddings_20241120_20250930"

SHARD_DIR_CANDIDATES = [
    Path("/Users/a964286/Documents/Projects/ClinTE/Clinical_Transformer_Emb/model_refactor/embedding_output/exp_round10_3lobs_formal_eval_commercial_ip"),
    Path("ClinTE/Clinical_Transformer_Emb/model_refactor/embedding_output/exp_round10_3lobs_formal_eval_commercial_ip"),
    Path("embedding_output/exp_round10_3lobs_formal_eval_commercial_ip"),
    Path("../embedding_output/exp_round10_3lobs_formal_eval_commercial_ip"),
    Path("../../embedding_output/exp_round10_3lobs_formal_eval_commercial_ip"),
]

def _npz_scalar(value):
    if isinstance(value, np.ndarray):
        if value.ndim == 0:
            return value.item()
        if value.size == 1:
            return value.reshape(()).item()
    return value

SHARD_SOURCE_DIR = next((path for path in SHARD_DIR_CANDIDATES if path.exists()), None)
if SHARD_SOURCE_DIR is None:
    searched = "\n".join(f"  - {path}" for path in SHARD_DIR_CANDIDATES)
    raise FileNotFoundError(
        "Could not find the saved shard directory. Update SHARD_DIR_CANDIDATES to match your local path.\n"
        f"Searched:\n{searched}"
    )

shard_files = sorted(SHARD_SOURCE_DIR.glob(f"{EXP_NAME}_commercial_ip_11M_*.npz"))
if not shard_files:
    shard_files = sorted(SHARD_SOURCE_DIR.glob("*.npz"))
if not shard_files:
    raise FileNotFoundError(f"No shard .npz files found in {SHARD_SOURCE_DIR}")

shard_summaries = []
for shard_file in shard_files:
    with np.load(shard_file, allow_pickle=False) as shard_npz:
        shard_summaries.append(
            {
                "file_path": shard_file,
                "shard_name": str(_npz_scalar(shard_npz["shard_name"])) if "shard_name" in shard_npz else shard_file.stem,
                "shard_index": int(_npz_scalar(shard_npz["shard_index"])) if "shard_index" in shard_npz else len(shard_summaries) + 1,
                "total_shards": int(_npz_scalar(shard_npz["total_shards"])) if "total_shards" in shard_npz else len(shard_files),
                "rows": int(shard_npz["embeddings"].shape[0]),
                "embedding_dim": int(shard_npz["embeddings"].shape[1]),
                "start_row": int(_npz_scalar(shard_npz["start_row"])) if "start_row" in shard_npz else None,
                "end_row": int(_npz_scalar(shard_npz["end_row"])) if "end_row" in shard_npz else None,
                "exp_name": str(_npz_scalar(shard_npz["exp_name"])) if "exp_name" in shard_npz else EXP_NAME,
                "model_type": str(_npz_scalar(shard_npz["model_type"])) if "model_type" in shard_npz else model_type,
            }
        )

shard_summaries = sorted(shard_summaries, key=lambda item: item["shard_index"])
expected_total_shards = shard_summaries[0]["total_shards"]
expected_embedding_dim = shard_summaries[0]["embedding_dim"]

assert len(shard_summaries) == expected_total_shards, (
    f"Expected {expected_total_shards} shards but found {len(shard_summaries)} in {SHARD_SOURCE_DIR}"
)
for expected_index, summary in enumerate(shard_summaries, start=1):
    assert summary["shard_index"] == expected_index, (
        f"Missing or out-of-order shard. Expected index {expected_index}, got {summary['shard_index']}"
    )
    assert summary["embedding_dim"] == expected_embedding_dim, (
        f"Embedding dim mismatch in {summary['file_path'].name}: "
        f"{summary['embedding_dim']} vs {expected_embedding_dim}"
    )

print(f"Shard source directory: {SHARD_SOURCE_DIR.resolve()}")
print(f"Shards found: {len(shard_summaries)}")
print(f"Embedding dim: {expected_embedding_dim}")
print(f"Target table: {EXPORT_PROJECT_ID}.{EXPORT_DATASET_ID}.{EXPORT_TABLE_NAME}")

total_rows_uploaded = 0
full_table_path = None

for shard_offset, summary in enumerate(shard_summaries):
    print("\n" + "=" * 70)
    print(
        f"Uploading shard {summary['shard_index']}/{expected_total_shards}: "
        f"{summary['shard_name']} ({summary['rows']:,} rows)"
    )

    with np.load(summary["file_path"], allow_pickle=False) as shard_npz:
        shard_embeddings = shard_npz["embeddings"].astype(np.float32, copy=False)
        shard_individual_ids = shard_npz["individual_ids"].astype(str).tolist()
        shard_index_dts = shard_npz["index_dts"].astype(str).tolist()

    assert shard_embeddings.shape[0] == len(shard_individual_ids) == len(shard_index_dts), (
        f"Row mismatch in {summary['file_path'].name}"
    )

    full_table_path = save_embeddings_to_bigquery(
        embeddings=shard_embeddings,
        individual_ids=shard_individual_ids,
        index_dts=shard_index_dts,
        project_id=EXPORT_PROJECT_ID,
        dataset_id=EXPORT_DATASET_ID,
        table_name=EXPORT_TABLE_NAME,
        exp_name=summary["exp_name"],
        model_type=summary["model_type"],
        if_exists="replace" if shard_offset == 0 else "append",
    )

    total_rows_uploaded += shard_embeddings.shape[0]
    print(f"Cumulative rows uploaded: {total_rows_uploaded:,}")

    del shard_embeddings, shard_individual_ids, shard_index_dts
    gc.collect()

print("\n" + "=" * 70)
print(f"All shard files uploaded to: {full_table_path}")
print(f"Total rows uploaded: {total_rows_uploaded:,}")
print("The four saved shard files have been combined logically in BigQuery.")

In [ ]:
# ============================================================================
# CLEANUP
# ============================================================================

del model, embeddings, df_cm, df_cm_sample
gc.collect()
cleanup_gpu_memory(verbose=True)
print("Done. All resources released.")

## 7. Unit Tests

Self-contained tests using synthetic data — no BigQuery or GPU required (CPU fallback).

| # | Test | What it validates |
|---|------|-------------------|
| 1 | `test_config_setup` | Config classes, experiment registry |
| 2 | `test_lazy_dataset` | LazyClinicalDatasetInference tensor shapes |
| 3 | `test_collate_fn` | Batch collation shapes and types |
| 4 | `test_model_forward_pass` | FlashAttentionTransformer output shape |
| 5 | `test_embedding_extractor` | EmbeddingExtractor hook captures correct dims |
| 6 | `test_generate_embeddings_e2e` | Full pipeline: synthetic data → embeddings |
| 7 | `test_load_model_roundtrip` | Save → load → weight equality |

In [15]:
# ============================================================================
# TEST UTILITIES — SYNTHETIC DATA FACTORY
# ============================================================================

def create_synthetic_data(n_samples: int = 20, n_days: int = 5) -> pd.DataFrame:
    """
    Create a synthetic DataFrame matching the production schema.
    All string fields use the '*'-separated day format.
    """
    rng = np.random.RandomState(42)
    rows = []
    for i in range(n_samples):
        dt_cnt = rng.randint(1, n_days + 1)
        ages = [str(rng.randint(120, 960)) for _ in range(dt_cnt)]
        genders = [str(rng.randint(1, 3)) for _ in range(dt_cnt)]
        lob_val = rng.choice(["Commercial", "Medicare", "Medicaid"])

        cd_days = []
        for _ in range(dt_cnt):
            n_codes = rng.randint(1, 10)
            cd_days.append(",".join(str(rng.randint(1, 1000)) for _ in range(n_codes)))

        target_days = []
        for _ in range(dt_cnt):
            n_tgt = rng.randint(0, 4)
            if n_tgt > 0:
                target_days.append(",".join(str(rng.randint(1, 500)) for _ in range(n_tgt)))
            else:
                target_days.append("")

        rows.append({
            "individual_id": f"MBR_{i:04d}",
            "index_dt": f"2023-{rng.randint(1, 13):02d}-{rng.randint(1, 29):02d}",
            "age_in_months": "*".join(ages),
            "gender_cd": "*".join(genders),
            "cd": "*".join(cd_days),
            "lob": lob_val,
            "dt_cnt": dt_cnt,
            "target": "*".join(target_days),
        })
    return pd.DataFrame(rows)


# Quick schema verification
_test_df = create_synthetic_data(3)
print("Synthetic data columns:", list(_test_df.columns))
print(f"Sample row keys: {list(_test_df.iloc[0].to_dict().keys())}")
del _test_df

Synthetic data columns: ['individual_id', 'index_dt', 'age_in_months', 'gender_cd', 'cd', 'lob', 'dt_cnt', 'target']
Sample row keys: ['individual_id', 'index_dt', 'age_in_months', 'gender_cd', 'cd', 'lob', 'dt_cnt', 'target']


In [62]:
# ============================================================================
# UNIT TESTS
# ============================================================================

import tempfile
import traceback


def test_config_setup():
    """Config classes instantiate with correct defaults; experiment registry works."""
    base = BaseConfig()
    assert base.len_dy == 200
    assert base.len_cd == 80
    assert base.embedding_size == 256
    assert base.target_cd_cnt == 6297

    flash = FlashAttentionConfig()
    assert flash.use_flash is True
    assert flash.use_rope is True
    assert flash.use_swiglu is True

    opt = OptimizeConfig(scheduler_type="linear", pos_weight_max=200)
    assert opt.scheduler_type == "linear"
    assert opt.pos_weight_max == 200

    configs = get_experiment_configs()
    assert "exp2b_flash_learned_pool" in configs
    moe_cfg, use_pool = configs["exp2b_flash_learned_pool"]
    assert moe_cfg is None, "exp2b should have no MoE"
    assert use_pool is True, "exp2b should use learned pooling"
    print("  test_config_setup PASSED")


def test_lazy_dataset():
    """LazyClinicalDatasetInference produces correct tensor shapes."""
    cfg = BaseConfig(len_dy=10, len_cd=5, target_cd_cnt=500)
    df = create_synthetic_data(n_samples=8, n_days=5)

    ds = LazyClinicalDatasetInference(df, cfg)
    assert len(ds) == 8

    item = ds[0]
    assert item["age"].shape == (10,)
    assert item["gender"].shape == (10,)
    assert item["lob"].shape == (10,)
    assert item["codes"].shape == (10, 5)
    assert isinstance(item["dt_cnt"], (int, np.integer))

    # Without target column
    df_no_tgt = df.drop(columns=["target"])
    ds2 = LazyClinicalDatasetInference(df_no_tgt, cfg)
    item2 = ds2[0]
    assert len(item2["target"]) == 10, "Dummy targets should have len_dy entries"
    print("  test_lazy_dataset PASSED")

def test_resolve_per_gpu_num_workers():
    """Multi-GPU worker resolution should preserve notebook-safe zero workers."""
    assert _resolve_per_gpu_num_workers(0, 4) == 0
    assert _resolve_per_gpu_num_workers(-1, 4) == 0
    assert _resolve_per_gpu_num_workers(1, 4) == 1
    assert _resolve_per_gpu_num_workers(4, 4) == 1
    assert _resolve_per_gpu_num_workers(8, 4) == 2
    print("  test_resolve_per_gpu_num_workers PASSED")
    
def test_collate_fn():
    """Collate function produces correct batch shapes and types."""
    cfg = BaseConfig(len_dy=10, len_cd=5, target_cd_cnt=500)
    df = create_synthetic_data(n_samples=8, n_days=5)

    ds = LazyClinicalDatasetInference(df, cfg)
    collate = create_collate_fn(cfg)
    loader = DataLoader(ds, batch_size=4, shuffle=False, collate_fn=collate)

    batch = next(iter(loader))
    assert batch["age"].shape == (4, 10), f"age: {batch['age'].shape}"
    assert batch["gender"].shape == (4, 10), f"gender: {batch['gender'].shape}"
    assert batch["lob"].shape == (4, 10), f"lob: {batch['lob'].shape}"
    assert batch["codes"].shape == (4, 10, 5), f"codes: {batch['codes'].shape}"
    assert batch["dt_cnt"].shape == (4,), f"dt_cnt: {batch['dt_cnt'].shape}"
    assert batch["target_multihot"].shape == (4, 10, 500), f"target: {batch['target_multihot'].shape}"
    print("  test_collate_fn PASSED")


def test_model_forward_pass():
    """FlashAttentionTransformer forward produces correct output shape."""
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = FlashAttentionConfig(
        len_dy=10, len_cd=5, embedding_size=32, nhid=64, nhead=4, nlayers=2,
        target_cd_cnt=500, cd_cnt=1000,
        use_learnt_att_pool=True, use_swiglu=True, use_rope=True, use_flash=True,
    )
    mdl = FlashAttentionTransformer(cfg).to(dev)
    mdl.eval()

    bs = 4
    # Input: [batch, len_dy, 1(age) + 1(gender) + 1(lob) + len_cd(codes)]
    x = torch.randint(0, 100, (bs, 10, 5 + 3)).float().to(dev)

    with torch.no_grad():
        out = mdl(x)
    assert out.shape == (bs, 10, 500), f"Output: {out.shape}"
    print(f"  test_model_forward_pass PASSED — output {out.shape}")

    del mdl
    if dev.type == "cuda":
        torch.cuda.empty_cache()


def test_embedding_extractor():
    """EmbeddingExtractor captures patient embeddings at correct dimensions."""
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = FlashAttentionConfig(
        len_dy=10, len_cd=5, embedding_size=32, nhid=64, nhead=4, nlayers=2,
        target_cd_cnt=500, cd_cnt=1000,
        use_learnt_att_pool=True, use_swiglu=True, use_rope=True, use_flash=True,
    )
    mdl = FlashAttentionTransformer(cfg).to(dev)
    mdl.eval()

    x = torch.randint(0, 100, (4, 10, 8)).float().to(dev)
    dt_cnt = [3, 5, 2, 7]

    with torch.no_grad():
        with EmbeddingExtractor(mdl) as ext:
            _ = mdl(x)
            embs = ext.get_patient_embedding(dt_cnt)

    assert embs.shape == (4, 32), f"Embedding: {embs.shape}"
    assert not torch.isnan(embs).any(), "NaN in embeddings"
    print(f"  test_embedding_extractor PASSED — {embs.shape}")

    del mdl
    if dev.type == "cuda":
        torch.cuda.empty_cache()


def test_generate_embeddings_e2e():
    """End-to-end: synthetic data -> generate_embeddings -> verify shape and values."""
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = FlashAttentionConfig(
        len_dy=10, len_cd=5, embedding_size=32, nhid=64, nhead=4, nlayers=2,
        target_cd_cnt=500, cd_cnt=1000,
        use_learnt_att_pool=True, use_swiglu=True, use_rope=True, use_flash=True,
    )
    mdl = FlashAttentionTransformer(cfg).to(dev)
    mdl.eval()

    df = create_synthetic_data(n_samples=12, n_days=5)

    embs, ids, dts = generate_embeddings(
        model=mdl, config=cfg, data=df, device=dev,
        id_column="individual_id", batch_size=4, num_workers=0,
        use_mixed_precision=False, verbose=False, multi_gpu=False,
    )

    assert embs.shape == (12, 32), f"Shape: {embs.shape}"
    assert len(ids) == 12
    assert len(dts) == 12
    assert not np.isnan(embs).any(), "NaN in embeddings"
    assert not np.isinf(embs).any(), "Inf in embeddings"
    assert ids[0] == "MBR_0000", f"First ID: {ids[0]}"
    print(f"  test_generate_embeddings_e2e PASSED — {embs.shape}")

    del mdl
    if dev.type == "cuda":
        torch.cuda.empty_cache()


def test_load_model_roundtrip():
    """Save checkpoint -> load_model_from_checkpoint -> verify weight equality."""
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = FlashAttentionConfig(
        len_dy=10, len_cd=5, embedding_size=32, nhid=64, nhead=4, nlayers=2,
        target_cd_cnt=500, cd_cnt=1000,
        use_learnt_att_pool=True, use_swiglu=True, use_rope=True, use_flash=True,
    )
    mdl = FlashAttentionTransformer(cfg).to(dev)

    with tempfile.TemporaryDirectory() as tmpdir:
        path = os.path.join(tmpdir, "test_model.pt")
        torch.save({
            "model_state_dict": mdl.state_dict(),
            "model_type": "FlashAttentionTransformer",
            "config": {
                "embedding_size": 32, "nhid": 64, "nhead": 4, "nlayers": 2,
                "dropout": 0.1, "use_learnt_att_pool": True,
                "use_swiglu": True, "use_rope": True, "use_flash": True,
                "len_dy": 10, "len_cd": 5, "target_cd_cnt": 500, "cd_cnt": 1000,
            },
            "moe_config": None,
        }, path)

        loaded, lcfg, _, use_mp, mtype = load_model_from_checkpoint(path, dev, verbose=False)

        assert mtype == "FlashAttentionTransformer"
        assert lcfg.embedding_size == 32
        assert use_mp is True

        for (n1, p1), (n2, p2) in zip(mdl.state_dict().items(), loaded.state_dict().items()):
            assert n1 == n2, f"Key mismatch: {n1} vs {n2}"
            assert torch.equal(p1.cpu(), p2.cpu()), f"Weight mismatch: {n1}"

    print("  test_load_model_roundtrip PASSED")

    del mdl, loaded
    if dev.type == "cuda":
        torch.cuda.empty_cache()

In [63]:
# ============================================================================
# RUN ALL TESTS
# ============================================================================

print("=" * 70)
print("RUNNING UNIT TESTS")
print("=" * 70)

tests = [
    test_resolve_per_gpu_num_workers,
    test_config_setup,
    test_lazy_dataset,
    test_collate_fn,
    test_model_forward_pass,
    test_embedding_extractor,
    test_generate_embeddings_e2e,
    test_load_model_roundtrip,
]

passed = 0
failed = 0
for fn in tests:
    try:
        fn()
        passed += 1
    except Exception as e:
        print(f"  FAILED: {fn.__name__} — {e}")
        traceback.print_exc()
        failed += 1

print(f"\n{'=' * 70}")
print(f"RESULTS: {passed} passed, {failed} failed out of {len(tests)}")
print(f"{'=' * 70}")

assert failed == 0, f"{failed} test(s) failed!"

RUNNING UNIT TESTS
  test_resolve_per_gpu_num_workers PASSED
  test_config_setup PASSED
LazyClinicalDatasetInference: 8 samples (lazy loading)
LazyClinicalDatasetInference: 8 samples (lazy loading)
  test_lazy_dataset PASSED
LazyClinicalDatasetInference: 8 samples (lazy loading)
  test_collate_fn PASSED
⚠️ Warning: head_dim=8 not optimal for xFormers.
   Recommended: 32, 64, or 128. Current: 8
⚠️ Warning: head_dim=8 not optimal for xFormers.
   Recommended: 32, 64, or 128. Current: 8


/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [37,0,0], thread: [0,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [37,0,0], thread: [1,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [37,0,0], thread: [2,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [37,0,0], thread: [3,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [37,0,0], thread: [4,0,0] Assertion 

  FAILED: test_model_forward_pass — CUDA error: CUBLAS_STATUS_EXECUTION_FAILED when calling `cublasSgemm( handle, opa, opb, m, n, k, &alpha, a, lda, b, ldb, &beta, c, ldc)`
⚠️ Warning: head_dim=8 not optimal for xFormers.
   Recommended: 32, 64, or 128. Current: 8
⚠️ Warning: head_dim=8 not optimal for xFormers.
   Recommended: 32, 64, or 128. Current: 8


Traceback (most recent call last):
  File "/var/tmp/ipykernel_3372453/622909435.py", line 24, in <module>
    fn()
  File "/var/tmp/ipykernel_3372453/2144171448.py", line 100, in test_model_forward_pass
    out = mdl(x)
  File "/opt/conda/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
  File "/home/jupyter/ClinTE/Clinical_Transformer_Emb/model_refactor/moe_flashattn_4_core.py", line 2441, in forward
    cd = self.daily_pooling(cd)  # [batch*len_dy, embedding_size]
  File "/opt/conda/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
  F

  FAILED: test_embedding_extractor — CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

⚠️ Warning: head_dim=8 not optimal for xFormers.
   Recommended: 32, 64, or 128. Current: 8
⚠️ Warning: head_dim=8 not optimal for xFormers.
   Recommended: 32, 64, or 128. Current: 8
  FAILED: test_generate_embeddings_e2e — CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNC

Traceback (most recent call last):
  File "/var/tmp/ipykernel_3372453/622909435.py", line 24, in <module>
    fn()
  File "/var/tmp/ipykernel_3372453/2144171448.py", line 117, in test_embedding_extractor
    mdl = FlashAttentionTransformer(cfg).to(dev)
  File "/opt/conda/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1381, in to
    return self._apply(convert)
  File "/opt/conda/lib/python3.10/site-packages/torch/nn/modules/module.py", line 933, in _apply
    module._apply(fn)
  File "/opt/conda/lib/python3.10/site-packages/torch/nn/modules/module.py", line 964, in _apply
    param_applied = fn(param)
  File "/opt/conda/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1367, in convert
    return t.to(
torch.AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other AP

AssertionError: 4 test(s) failed!